# KOTFlash — полный прогон на Tesla T4

Этот ноутбук поднимает отдельное Python 3.12-окружение, устанавливает совместимый стек PyTorch 2.6.0 + CUDA 12.4 + Triton 3.2.x, клонирует рабочую ветку KOTFlash и запускает CPU/API/GPU correctness suite вместе с attention, decode и synthetic decoder-block benchmarks.

**Перед запуском:** в Colab выбери `Runtime → Change runtime type → T4 GPU`. Скрипт намеренно остановится, если обнаружит не Tesla T4 / SM75 или несовместимую версию библиотек. Результаты — это attention/decoder-block benchmarks, а не обещание ускорения полной LLM.

Опциональная ячейка в конце отдельно собирает CUDA C++ append extension. Она требует доступного `nvcc` и может быть пропущена.

In [ ]:
# Install uv in the base Colab runtime; the project dependencies stay isolated below.
%pip -q install uv
!uv python install 3.12.16
!uv venv --python 3.12.16 /content/kotflash-env

PY = "/content/kotflash-env/bin/python"
print("Isolated interpreter:", PY)

In [ ]:
# Get the exact feature branch containing the T4 kernels and validation suite.
from pathlib import Path
import subprocess

REPO = Path("/content/KOTFlash")
BRANCH = "feature/t4-performance-rebased"
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--branch", BRANCH, "--depth", "1",
        "https://github.com/AitiKot/KOTFlash.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)

subprocess.run([PY, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([
    PY, "-m", "pip", "install", "--index-url",
    "https://download.pytorch.org/whl/cu124", "torch==2.6.0"
], check=True)
subprocess.run([PY, "-m", "pip", "install", "numpy", "pytest", "build", "ninja"], check=True)
subprocess.run([PY, "-m", "pip", "install", "-e", ".[t4,test]"], cwd=REPO, check=True)
print("Installed KOTFlash from:", REPO)

In [ ]:
# Verify the interpreter and GPU before spending time on the full benchmark suite.
import subprocess

check = r'''import torch, triton, kotflash
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA unavailable")
print("Capability:", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None)
print("KOTFlash:", kotflash.__file__)
assert torch.cuda.is_available(), "Enable a GPU runtime in Colab"
assert torch.cuda.get_device_capability(0) == (7, 5), "This notebook is for SM75 / Tesla T4"
assert "T4" in torch.cuda.get_device_name(0), "Refusing to label a non-T4 GPU run as T4 validation"
assert torch.__version__.split("+")[0] == "2.6.0" and torch.version.cuda == "12.4"
assert triton.__version__.split(".")[:2] == ["3", "2"]
'''
subprocess.run([PY, "-c", check], check=True)

In [ ]:
# Full physical-T4 validation: compileall, 107+ CPU/GPU tests, attention/decode/block benchmarks.
# The script writes JSON and pytest logs into /content/KOTFlash/artifacts/.
import os

env = os.environ.copy()
env["PATH"] = "/content/kotflash-env/bin:" + env.get("PATH", "")
subprocess.run(["bash", "benchmarks/run_t4_validation.sh"], cwd=REPO, env=env, check=True)
print("Validation artifacts:", sorted(str(p) for p in (REPO / "artifacts").glob("t4-validation-*")))

In [ ]:
# Optional: build and test the first-party CUDA C++ K/V append extension.
# Skip this cell when nvcc is unavailable; the standard Triton validation above does not need it.
nvcc = subprocess.run(["bash", "-lc", "command -v nvcc"], capture_output=True, text=True)
if nvcc.returncode != 0:
    print("nvcc not found. Optional CUDA C++ append test skipped.")
else:
    subprocess.run([PY, "-m", "pip", "install", "-e", ".[cuda-append]"], cwd=REPO, check=True)
    env = os.environ.copy()
    env["KOTFLASH_TEST_CUDA_APPEND"] = "1"
    env["PATH"] = "/content/kotflash-env/bin:" + env.get("PATH", "")
    subprocess.run([PY, "-m", "pytest", "-q", "tests/test_cuda_append.py"], cwd=REPO, env=env, check=True)

## Что сохранять после запуска

Папка `artifacts/t4-validation-*` содержит `pytest.txt`, sweep-результаты для attention длиной 128–2048, decode Q=1/KV=2048 и synthetic decoder-block benchmark. При публикации результатов указывай модель GPU, PyTorch/CUDA/Triton и конкретную форму тензоров. Не смешивай attention-only latency с полной скоростью генерации или обучения модели.